<a href="https://colab.research.google.com/github/sithmi4/Statistical-Learning-e23207/blob/main/E23207_Bayesian_Inference_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

This notebook presents a complete solution for Structural Health Monitoring (SHM) using bounded Bayesian grid updates to estimate the remaining stiffness efficiency factor $\theta \in (0, 1]$ of a structural component under non-linear, multiplicative log-normal sensor noise.

## 1. Prior Belief Boundaries

### **Analytical Prior Expectation**
The initial prior state is modeled using a Beta distribution over $(0, 1]$: $\Theta \sim \text{Beta}(\alpha, \beta)$ where $\alpha = 8$ and $\beta = 1.5$.

The analytical expected value of a Beta-distributed random variable is given by:
$$\mathbb{E}[\Theta^{(0)}] = \frac{\alpha}{\alpha + \beta} = \frac{8}{8 + 1.5} = \frac{8}{9.5} = \frac{16}{19} \approx 0.8421$$

### **Engineering Justification**
This distribution serves as an appropriate initial prior for an engineering component assumed to be healthy for several physical reasons:
1. **Bounded Domain Enforcement:** Structural stiffness efficiency $\theta$ is physically bounded within $(0, 1]$, where $\theta = 1.0$ represents a pristine component and $\theta \to 0$ signifies critical degradation. The Beta distribution naturally operates over $[0, 1]$, enforcing physical boundaries without assigning probability mass to non-physical negative stiffness or >100% health.
2. **Optimistic Health Prior:** With shape parameters $\alpha = 8$ and $\beta = 1.5$, the density function is heavily left-skewed with a mode at $\frac{\alpha - 1}{\alpha + \beta - 2} = \frac{7}{7.5} \approx 0.9333$. This reflects structural engineering baseline assumptions: components deployed in service are assumed healthy prior to collecting sensor evidence.
3. **Continuous Uncertainty Representation:** While favoring high efficiency, the Beta distribution maintains non-zero probability density across the entire domain, allowing sensor data to smoothly update the distribution if structural cracking exists.

In [1]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Define grid domain over the physical interval [0.01, 1.0]
theta_grid = np.linspace(0.01, 1.0, 500)

# Initial Prior Density: Beta(8, 1.5)
prior_density = stats.beta.pdf(theta_grid, a=8, b=1.5)

# Analytical Expectation
E_theta_0 = 8 / (8 + 1.5)

# Plot Initial Prior Density using Plotly
fig_prior = go.Figure()
fig_prior.add_trace(go.Scatter(
    x=theta_grid,
    y=prior_density,
    mode='lines',
    name='Initial Prior Density Beta(8, 1.5)',
    line=dict(color='royalblue', width=3)
))

fig_prior.add_vline(
    x=E_theta_0,
    line_dash="dash",
    line_color="firebrick",
    line_width=2,
    annotation_text=f"Expected Prior E[θ] = {E_theta_0:.4f}",
    annotation_position="top left"
)

fig_prior.update_layout(
    title="Initial Prior Density Function: Beta(8, 1.5)",
    xaxis_title="Remaining Stiffness Efficiency Factor (θ)",
    yaxis_title="Probability Density f_Θ(θ)",
    template="plotly_white",
    width=850,
    height=450
)

fig_prior.show()

## 2. Structural Likelihood Formulation

### **Single Measurement Likelihood $L(y_k \mid \theta)$**
The sensor model at inspection step $k$ is given by:
$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

Taking the natural logarithm of both sides:
$$\ln(y_k) = \ln(\theta \cdot K_{\text{nominal}}) + \epsilon_k$$

Because $\epsilon_k \sim \mathscr{N}(0, \sigma^2)$, the term $\ln(y_k)$ follows a Gaussian distribution with mean $\mu = \ln(\theta \cdot K_{\text{nominal}})$ and variance $\sigma^2$. Consequently, the observation $y_k$ follows a **Log-Normal distribution** with parameter $\text{scale} = \theta \cdot K_{\text{nominal}}$ and shape parameter $\sigma$.

Using change of variables $f_{Y_k}(y_k) = f_{\ln Y_k}(\ln y_k) \left| \frac{d \ln y_k}{d y_k} \right| = f_{\ln Y_k}(\ln y_k) \cdot \frac{1}{y_k}$, the single measurement likelihood contribution is:
$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{\left( \ln y_k - \ln(\theta \cdot K_{\text{nominal}}) \right)^2}{2\sigma^2} \right)$$

---

### **Joint Likelihood Function for Running History Vector $\mathbf{y}^{(k)}$**
Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)^T$ represent the sensor history up to step $k$. Assuming conditional independence across inspection steps given $\theta$, the joint likelihood function is:
$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k L(y_i \mid \theta) = \left( \prod_{i=1}^k \frac{1}{y_i \sigma \sqrt{2\pi}} \right) \exp\left( -\frac{1}{2\sigma^2} \sum_{i=1}^k \left( \ln y_i - \ln(\theta \cdot K_{\text{nominal}}) \right)^2 \right)$$

## 3. Mathematical Formulation of the Non-Conjugate Grid Update

### **Non-Conjugacy Explanation**
A prior distribution $f(\theta)$ is conjugate to a likelihood $L(y \mid \theta)$ if the resulting posterior $f(\theta \mid y)$ remains in the same algebraic family as the prior.

* The initial prior is a **Beta distribution**: $f_0(\theta) \propto \theta^{\alpha-1} (1-\theta)^{\beta-1}$.
* The structural likelihood is **Log-Normal**: $L(y_k \mid \theta) \propto \exp\left(-\frac{(\ln y_k - \ln \theta - \ln K_{\text{nominal}})^2}{2\sigma^2}\right)$.

Because $\theta$ appears non-linearly inside a logarithm within the exponent of a normal noise model, the product $f(\theta) \cdot L(y_k \mid \theta)$ cannot be simplified algebraically into an updated Beta distribution. Consequently, the normalization integral $Z_k = \int_{0}^{1} f_{k-1}(	heta) L(y_k \mid \theta) d\theta$ lacks a closed analytical solution, requiring **numerical grid updating**.

---

### **Recursive Posterior Relationship**
In sequential Bayesian updating, the posterior from step $k-1$ acts as the prior for step $k$. Up to a normalization constant, the update equation is:
$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

Expressed with the normalization constant $Z_k$:
$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})}{\int_{0}^{1} L(y_k \mid s) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(s \mid \mathbf{y}^{(k-1)}) \, ds}$$

## 4. Running Point Estimates

Because a closed-form analytical density is unavailable, point estimators are defined via numerical integration over the bounded domain $(0, 1]$ at step $k$:

### **1. Running Posterior Mean (Bayes Estimate)**
Under a minimum mean squared error loss function, the optimal point estimate is the expected value of the current bounded posterior:
$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \mathbb{E}\left[ \Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)} \right] = \int_{0}^{1} \theta \cdot f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta$$

### **2. Running Maximum A Posteriori (MAP Estimate)**
The MAP estimate represents the mode (highest density point) of the posterior density over $(0, 1]$:
$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \arg\max_{\theta \in (0, 1]} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$$

## 5. Algorithmic Grid Approximation and Normalization

To maintain the distribution without analytical conjugacy, we discretize the domain $(0, 1]$ onto a fine grid $\boldsymbol{\theta} = [\theta_1, \theta_2, \dots, \theta_M]$.

### **Step-by-Step Numerical Procedure:**

1. **Grid Discretization:**
   Define $M$ equally spaced grid points across $[\theta_{\min}, \theta_{\max}] = [0.01, 1.0]$:
   $$\theta_m = \theta_{\min} + (m-1)\Delta\theta, \quad \Delta\theta = \frac{\theta_{\max} - \theta_{\min}}{M-1}, \quad m = 1, 2, \dots, M$$

2. **Prior Initialization & Normalization:**
   Evaluate initial prior values across the grid $\mathbf{P}_0 = [f_0(\theta_1), \dots, f_0(\theta_M)]$ and normalize using the composite trapezoidal rule:
   $$Z_0 = \text{trapezoid}(\mathbf{P}_0, \boldsymbol{\theta}) = \sum_{m=1}^{M-1} \frac{P_0(\theta_m) + P_0(\theta_{m+1})}{2} \Delta\theta, \qquad \mathbf{P}_0 \leftarrow \frac{\mathbf{P}_0}{Z_0}$$

3. **Sequential Updating & Normalization (at step $k = 1, 2, \dots, n$):**
   * Evaluate likelihood array: $\mathbf{L}_k = [L(y_k \mid \theta_1), \dots, L(y_k \mid \theta_M)]$.
   * Compute unnormalized posterior array: $\mathbf{\tilde{P}}_k = \mathbf{P}_{k-1} \odot \mathbf{L}_k$.
   * Calculate normalizing constant: $Z_k = \text{trapezoid}(\mathbf{\tilde{P}}_k, \boldsymbol{\theta})$.
   * Normalize posterior array: $\mathbf{P}_k = \frac{\mathbf{\tilde{P}}_k}{Z_k}$.

4. **Point Estimator Computation:**
   * **Posterior Mean:** $\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \text{trapezoid}(\boldsymbol{\theta} \odot \mathbf{P}_k, \boldsymbol{\theta})$.
   * **MAP Estimate:** $\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \theta_{m^*}$, where $m^* = \arg\max_{m} P_k(\theta_m)$.

In [2]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go
from scipy.integrate import trapezoid

# Seed for reproducible noisy sensor data generation
np.random.seed(42)

# System Parameters
theta_true = 0.68        # Damaged structural stiffness state (68% efficiency)
K_nominal = 50.0         # Pristine baseline stiffness (kN/mm)
sigma = 0.15             # Log-space sensor noise
n_steps = 15             # Total inspection steps
M_grid = 1000            # Discretization grid points

# 1. Define bounded discretization grid
theta_grid = np.linspace(0.01, 1.0, M_grid)

# 2. Initialize Prior: Beta(8, 1.5)
current_posterior = stats.beta.pdf(theta_grid, a=8, b=1.5)
current_posterior /= trapezoid(current_posterior, theta_grid)

# Milestones tracking
milestones = [0, 1, 2, 5, 10, 15]
posterior_history = {0: current_posterior.copy()}

# Tracking arrays for estimators and observations
bayes_history = [trapezoid(theta_grid * current_posterior, theta_grid)]
map_history = [theta_grid[np.argmax(current_posterior)]]
observed_readings = []

# 3. Sequential Monitoring Loop
for k in range(1, n_steps + 1):
    # Simulate noisy log-normal observation
    noise = np.random.normal(0, sigma)
    y_k = (theta_true * K_nominal) * np.exp(noise)
    observed_readings.append(y_k)

    # Compute Likelihood array across theta grid
    likelihood = stats.lognorm.pdf(y_k, s=sigma, scale=theta_grid * K_nominal)

    # Unnormalized posterior update
    unnormalized_posterior = current_posterior * likelihood

    # Normalization using trapezoidal integration
    Z_k = trapezoid(unnormalized_posterior, theta_grid)
    current_posterior = unnormalized_posterior / Z_k

    # Point estimate updates
    bayes_est = trapezoid(theta_grid * current_posterior, theta_grid)
    map_est = theta_grid[np.argmax(current_posterior)]

    bayes_history.append(bayes_est)
    map_history.append(map_est)

    if k in milestones:
        posterior_history[k] = current_posterior.copy()

# =====================================================================
# VISUALIZATION 1: POSTERIOR DENSITY PROFILE EVOLUTION AT MILESTONES
# =====================================================================
fig1 = go.Figure()
colors = ['#7f7f7f', '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

for idx, step in enumerate(milestones):
    dens = posterior_history[step]
    label = "Step 0 (Initial Prior)" if step == 0 else f"Step {step} (y_{step} = {observed_readings[step-1]:.2f})"
    dash_style = 'dash' if step == 0 else 'solid'
    fig1.add_trace(go.Scatter(
        x=theta_grid, y=dens, mode='lines',
        name=label,
        line=dict(width=2.5 if step in [0, 15] else 1.8, color=colors[idx], dash=dash_style)
    ))

fig1.add_vline(
    x=theta_true, line_dash="dot", line_color="black", line_width=2.5,
    annotation_text=f"True Structural State (θ_true = {theta_true})",
    annotation_position="top left"
)

fig1.update_layout(
    title="1. Progression of Bounded Posterior Densities across Inspection Milestones",
    xaxis_title="Remaining Structural Stiffness Efficiency Factor (θ)",
    yaxis_title="Probability Density f(θ | y^(k))",
    template="plotly_white",
    width=900, height=500,
    legend=dict(yanchor="top", y=0.98, xanchor="left", x=0.02, bgcolor="rgba(255,255,255,0.8)")
)
fig1.show()

# =====================================================================
# VISUALIZATION 2: ESTIMATOR CONVERGENCE TIMELINE (STEPS 0 TO 15)
# =====================================================================
steps_array = np.arange(0, n_steps + 1)

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=steps_array, y=bayes_history, mode='lines+markers',
    name='Running Posterior Mean (Bayes)',
    line=dict(color='crimson', width=2.5), marker=dict(size=7)
))
fig2.add_trace(go.Scatter(
    x=steps_array, y=map_history, mode='lines+markers',
    name='Running MAP Estimate',
    line=dict(color='darkblue', width=2, dash='dash'), marker=dict(size=6, symbol='square')
))
fig2.add_hline(
    y=theta_true, line_dash="dot", line_color="green", line_width=2.5,
    annotation_text=f"True Damage State (θ_true = {theta_true})",
    annotation_position="bottom right"
)

fig2.update_layout(
    title="2. Convergence Timeline of Point Estimators to True Damage State",
    xaxis_title="Inspection Time Step (k)",
    yaxis_title="Stiffness Efficiency Estimate (θ̂)",
    template="plotly_white",
    width=900, height=480,
    xaxis=dict(dtick=1),
    legend=dict(yanchor="top", y=0.95, xanchor="right", x=0.98, bgcolor="rgba(255,255,255,0.8)")
)
fig2.show()

## 6. Performance Tracking and Degradation Convergence Analysis

### **Convergence Behavior & Prior Overcoming:**
* **Initial Prior Bias:** At step $0$, the system begins with an optimistic Beta(8, 1.5) prior centered at $\mathbb{E}[\Theta^{(0)}] \approx 0.8421$ with a MAP mode at $0.9333$.
* **Rapid Transition (Steps 1–3):** As noisy measurements arrive from the damaged component ($\theta_{\text{true}} = 0.68$), the log-normal likelihood exerts immediate downward pressure on the posterior. By step $2$, the MAP estimate rapidly drops to $\sim 0.7433$.
* **Convergence Threshold:** Within **3 to 5 sensor readings**, the continuous observation stream overcomes the optimistic prior. Both point estimators settle tightly around $\theta_{\text{true}} = 0.68$. By step $15$, the Bayes estimate converges to **$0.6886$** and the MAP estimate to **$0.6868$**.

---

### **Structural Safety & Risk Threshold Implications:**
1. **Variance Reduction:** As sequential observations accumulate ($k = 0 \to 15$), the posterior distribution narrows significantly around $\theta = 0.68$, drastically reducing epistemic uncertainty.
2. **Credible Upper Bounds:** Structural safety decisions rely on credible upper limits (e.g., $95\%$ upper bound). When the posterior is wide (steps 0–2), high uncertainty prevents distinguishing real structural damage from transient sensor noise. As density curves narrow, upper tail density above critical safety thresholds (e.g., $\theta > 0.75$) drops to near zero.
3. **Engineering Decision Making:** The narrowing density confirms that observed stiffness reductions are caused by actual structural degradation rather than sensor drift, giving engineers high confidence to trigger timely maintenance before failure.